# Prerequisites

In [ ]:
# get data for labs
!wget -nc -O around_the_world_in_80_days.txt https://www.gutenberg.org/ebooks/103.txt.utf-8

# 1. Word Count

Instructions:  
For each cell marked "double-click and add explanation here" please answer the question in your own words.  
In the section where you complete the code to perform basic nlp text cleaning and exploration tasks, the goal is to chain all of the transformations together in a single function. For learning and exploration purposes, it is acceptable to have each step seperate, but the last cell in this section should be one function with all transformations chained together.  
For steps c and f, it is acceptable to use your favorite chatbot to generate a list of common stop words (c) and punctuation (e) for use in the code. As these are common steps in nlp/text processing tasks, there are pleanty of libraries to help with this such as nltk, but there is no need to import extra dependencies for this lab unless you are already familiar with working with them.

In [1]:
# start a spark session and create spark context for making rdd
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("word_count") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
# Defind the rdd
rdd = sc.textFile('103.txt')
# rdd_2 = sc.textFile('/content/le_tour_du_monde_en_quatre_vingts_jours.txt')

In [3]:
# view the first x lines of the rdd
rdd.take(20)

['The Project Gutenberg eBook of Around the World in Eighty Days',
 '    ',
 'This eBook is for the use of anyone anywhere in the United States and',
 'most other parts of the world at no cost and with almost no restrictions',
 'whatsoever. You may copy it, give it away or re-use it under the terms',
 'of the Project Gutenberg License included with this eBook or online',
 'at www.gutenberg.org. If you are not located in the United States,',
 'you will have to check the laws of the country where you are located',
 'before using this eBook.',
 '',
 'Title: Around the World in Eighty Days',
 '',
 'Author: Jules Verne',
 '',
 'Translator: George M. Towle',
 '',
 '',
 '        ',
 'Release date: January 1, 1994 [eBook #103]',
 '                Most recently updated: October 29, 2024']

In [4]:
# example lambda function
words = rdd.flatMap(lambda lines: lines.split(' '))

In [6]:
# Note and explain the output of the below command
words

PythonRDD[3] at RDD at PythonRDD.scala:59

double-click and add explanation here

<ADD EXPLANATION HERE>

In [ ]:
# Note and explain the output of the following command, focusing on the difference with the
# above command
words.collect()

double-click and add explanation here

In [ ]:
# nicer print
for w in words.collect():
    print(w)

In [6]:
# Print first x words
words.take(20)

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 '',
 '',
 '',
 '',
 'This',
 'eBook',
 'is',
 'for']

In [ ]:
# Use cell magic command to help understand what the rdd.flatMap function is doing in the next cell.
# Insert a text/markdown cell and explain in your own words.

In [ ]:
# Initialize a word counter by creating a tuple with word and cound of 1
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word, 1))

for w in words.collect():
    print(w)

In [16]:
# a. count the occurence of each word
word_counts = words.reduceByKey(lambda a, b: a + b)
print(word_counts.takeOrdered(50, key=lambda x: -x[1]))

[('the', 4316), ('', 2192), ('of', 1875), ('and', 1793), ('to', 1690), ('a', 1261), ('in', 991), ('was', 974), ('his', 804), ('he', 720), ('at', 576), ('with', 550), ('not', 500), ('had', 484), ('The', 482), ('on', 482), ('that', 460), ('which', 414), ('for', 407), ('as', 396), ('by', 377), ('Mr.', 371), ('Fogg', 331), ('it', 322), ('be', 307), ('from', 302), ('were', 300), ('this', 292), ('is', 288), ('would', 266), ('have', 259), ('you', 243), ('an', 222), ('Passepartout', 219), ('but', 216), ('Phileas', 214), ('He', 208), ('I', 207), ('or', 185), ('they', 183), ('him', 181), ('who', 179), ('are', 169), ('been', 167), ('said', 155), ('her', 146), ('their', 145), ('It', 139), ('if', 138), ('could', 136)]


In [17]:
# b. a common first step in text analysis, change all capital letters to lower case
words_lower = word_counts.map(lambda x: (x[0].lower(), x[1])).reduceByKey(lambda a, b: a + b)
print(words_lower.takeSample(True, 20))

[('roughly', 1), ('forty-five', 3), ('morning', 16), ('5th.', 1), ('contents', 1), ('mast', 2), ('again', 20), ('smeared', 1), ('jersey', 2), ('bombay?”', 2), ('performer', 1), ('hastily.', 1), ('remark.', 1), ('palatial', 1), ('lost!', 1), ('sugar', 2), ('government,', 2), ('companions,', 1), ('might,', 2), ('deliverers,', 1)]


In [33]:
# c. eliminate the stop words.
stop_words = ["the", "", "of", "and", "to", "a", "in", "was", "his", "he", "at", "with", "not", "had", "The", "on", "that", "which", "for", "as", "by", "it", "be", "from", "were", "this", "is", "would", "have", "an", "but", "or" ]
filtered_words = words_lower.filter(lambda x: x[0] not in stop_words)
print(filtered_words.takeOrdered(50, key=lambda x: -x[1]))

[('mr.', 373), ('fogg', 365), ('you', 280), ('phileas', 250), ('passepartout', 239), ('they', 232), ('i', 207), ('him', 183), ('who', 182), ('are', 175), ('if', 170), ('been', 167), ('her', 157), ('said', 157), ('their', 147), ('all', 141), ('could', 139), ('no', 137), ('one', 133), ('its', 133), ('fogg,', 132), ('fix', 129), ('did', 126), ('passepartout,', 121), ('so', 118), ('when', 117), ('more', 116), ('“i', 115), ('upon', 113), ('what', 112), ('will', 111), ('about', 108), ('there', 107), ('only', 102), ('some', 101), ('after', 101), ('two', 98), ('up', 95), ('she', 95), ('we', 93), ('out', 92), ('now', 92), ('hundred', 92), ('my', 91), ('replied', 89), ('himself', 87), ('project', 87), ('without', 86), ('into', 84), ('thousand', 83)]


In [34]:
# d. sort in alphabetical order
sorted_alpha = filtered_words.sortBy(lambda x: x[0], ascending=True)
print(sorted_alpha.take(50))

[('#103]', 1), ('#516,', 1), ('$5,000)', 1), ('&c.,', 1), ('($1', 1), ('(862)', 1), ('(a)', 1), ('(and', 1), ('(any', 1), ('(b)', 1), ('(c)', 1), ('(does', 1), ('(if', 1), ('(japan),', 1), ('(or', 3), ('(saturday,', 1), ('(sort', 1), ('(sunday)', 1), ('(trademark/copyright)', 1), ('(www.gutenberg.org),', 1), ('(“the', 1), ('***', 4), ('+1', 1), ('-', 3), ('-------', 1), ('.', 3), ('.....', 1), ('........', 1), ('.........', 1), ('.............', 2), ('.................', 1), ('...................', 1), ('....................', 1), ('............................................', 1), ('07042,', 1), ('1,', 1), ('1,000.', 1), ('1,170,', 1), ('1.', 1), ('1.a.', 1), ('1.b.', 1), ('1.c', 1), ('1.c.', 1), ('1.d.', 1), ('1.e', 1), ('1.e.', 1), ('1.e.1', 3), ('1.e.1.', 2), ('1.e.2.', 1), ('1.e.3.', 1)]


In [35]:
# e. sort descending by word frequency
print(filtered_words.takeOrdered(50, key=lambda x: -x[1]))

[('mr.', 373), ('fogg', 365), ('you', 280), ('phileas', 250), ('passepartout', 239), ('they', 232), ('i', 207), ('him', 183), ('who', 182), ('are', 175), ('if', 170), ('been', 167), ('her', 157), ('said', 157), ('their', 147), ('all', 141), ('could', 139), ('no', 137), ('one', 133), ('its', 133), ('fogg,', 132), ('fix', 129), ('did', 126), ('passepartout,', 121), ('so', 118), ('when', 117), ('more', 116), ('“i', 115), ('upon', 113), ('what', 112), ('will', 111), ('about', 108), ('there', 107), ('only', 102), ('some', 101), ('after', 101), ('two', 98), ('up', 95), ('she', 95), ('we', 93), ('out', 92), ('now', 92), ('hundred', 92), ('my', 91), ('replied', 89), ('himself', 87), ('project', 87), ('without', 86), ('into', 84), ('thousand', 83)]


In [36]:
# f. remove punctuations and blank spaces
import string

words_stripped = sorted_freq.map(lambda x: (x[0].strip(string.punctuation), x[1]))
words_no_blanks = words_stripped.filter(lambda x: x[0] not in ["", " "])
final_clean_words = words_no_blanks.reduceByKey(lambda a, b: a + b)
sorted_alpha = final_clean_words.sortBy(lambda x: x[0], ascending=True)
print(sorted_alpha.take(50))

[('07042', 1), ('1', 5), ('1,000', 1), ('1,170', 1), ('1.a', 1), ('1.b', 1), ('1.c', 2), ('1.d', 1), ('1.e', 2), ('1.e.1', 5), ('1.e.2', 1), ('1.e.3', 1), ('1.e.4', 1), ('1.e.5', 1), ('1.e.6', 1), ('1.e.7', 3), ('1.e.8', 4), ('1.e.9', 3), ('1.f', 1), ('1.f.1', 1), ('1.f.2', 1), ('1.f.3', 5), ('1.f.4', 1), ('1.f.5', 1), ('1.f.6', 1), ('103', 1), ('10th', 1), ('11', 1), ('11.40', 1), ('117', 2), ('11th', 6), ('11th?”', 1), ('12th', 3), ('13', 2), ('13th', 3), ('14th', 4), ('158½', 1), ('15th', 1), ('16th', 1), ('1756', 1), ('17th', 2), ('1814', 1), ('1825', 1), ('1837', 1), ('1839', 1), ('1842', 1), ('1843', 1), ('1845', 1), ('1849—a', 1), ('1853', 1)]


# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be able to explain each line while respecting the pep8 style guide of 79 characters or less per line!

In [ ]:
 # Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30),
("TD", 35), ("Brooke", 25)])

# Try to undestand what this code does (line by line)
# Use map and reduceByKey transformations with their lambda
# Expressions to aggregate then compute average
agesRDD = (dataRDD
  # Maps to (name, (age, 1)) to prepare for summing ages and counting records.
  .map(lambda x: (x[0], (x[1], 1)))
  # Groups by name, adding the ages together and summing the total count.
  .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
  # Divides total age by total count to calculate the average age per person.
  .map(lambda x: (x[0], x[1][0]/x[1][1])))

# Where to go from here

Further exploration for students who complete the lab before the end of the session or want to go further.

- perform eda on the original french version of the [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two
- recomplete the exercises using a the docker install
- install java and spark directly onto host machine and either rexplore this notebook or perform eda on other data sets
- write a simple python timer function for seeing how quickly your rdd runs as written. change the order of the steps in order to make the rdd run as optimally as possible